# Basic Run - He into Fe

The complete OpenTRIM Python workflow: configure a simulation, run it,
read the results through `Info`, and plot the vacancy depth profile.

2 MeV He ions into a 4 μm Fe slab, 200 depth bins.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import opentrim
print('opentrim', opentrim.__version__)

## 1. Configure

In [ ]:
config = opentrim.Config()

# Ion beam: 2 MeV He
config.IonBeam.ion = opentrim.Element('He')
config.IonBeam.energy_distribution.center = 2e6  # eV

# Variable flight path - important for light ions
config.Transport.flight_path_type = opentrim.FlightPath.Variable

# Target: 4 μm Fe slab
mat = opentrim.Material()
mat.id = 'Fe'
mat.density = 7.874
atom = opentrim.Atom()
atom.element = opentrim.Element('Fe')
atom.X = 1.0
atom.Ed = 40.0
atom.El = 3.0
atom.Es = 4.28
atom.Er = 40.0
atom.Rc = 0.946
mat.composition.append(atom)
config.Target.materials.append(mat)

L = 4000
region = opentrim.Region()
region.id = 'slab'
region.material_id = 'Fe'
region.origin = [0, 0, 0]
region.size = [L,L,L]
config.Target.regions.append(region)

config.Target.cell_count = [200, 1, 1]
config.Target.size = [L,L,L]

# Physics
config.Simulation.electronic_stopping = opentrim.Stopping.SRIM13
config.Simulation.simulation_type = opentrim.SimulationType.FullCascade

# Run
config.Output.outfilename = 'He_2MeV_Fe'
config.Run.max_no_ions = 10000
config.Run.threads = 4
config.Run.seed = 42

config.validate()
config

## 2. Run (Mode A - non-blocking)

In [ ]:
sim = opentrim.Driver()
sim.init(config)

sim.run()                       # returns immediately
print('ions so far:', sim.ion_count())
sim.wait()                      # block until done
print('done, total ions:', sim.ion_count())

## 3. Read results

In [ ]:
info = opentrim.Info(sim)
info                            # collapsible tree in Jupyter

In [ ]:
# Grid: x-axis cell edges in nm -> cell centers
x_edges = info['target']['grid']['X']
x_centers = 0.5 * (x_edges[:-1] + x_edges[1:])

# Vacancies per ion, with error bars.  shape [natoms, Nx, Ny, Nz]
v, dv = info['tally']['damage_events']['Vacancies']
v_depth = v.sum(axis=0)[:, 0, 0]      # sum over atom species
dv_depth = np.sqrt((dv ** 2).sum(axis=0))[:, 0, 0]

## 4. Plot

In [ ]:
plt.figure(figsize=(8, 4))
plt.errorbar(x_centers, v_depth, yerr=dv_depth, fmt='-', capsize=2)
plt.xlabel('Depth (nm)')
plt.ylabel('Vacancies per ion')
plt.title(f'He 2 MeV -> Fe  (N={sim.ion_count():,} ions)')
plt.tight_layout()
plt.show()